In [5]:
import pandas as pd

In [6]:
data_path = '/content/drive/MyDrive/quickmart_data'

In [7]:
customers = pd.read_csv(f'{data_path}/customers.csv')
products = pd.read_csv(f'{data_path}/products.csv')
sales = pd.read_csv(f'{data_path}/sales.csv')
stores = pd.read_csv(f'{data_path}/stores.csv')


In [8]:
customers.head()

,customer_id,customer_name,email,city,state,customer_segment
0,C000001,David Okoro,customer000001@quickmart.example,Ikeja,Lagos,Mass Market
1,C000002,Peter Garba,customer000002@quickmart.example,Maiduguri,Borno,Mass Market
2,C000003,Hauwa Ogunleye,customer000003@quickmart.example,Port Harcourt,Rivers,Mass Market
3,C000004,Chinedu Okafor,customer000004@quickmart.example,Maiduguri,Borno,Mass Market
4,C000005,Chinedu Musa,customer000005@quickmart.example,Jos,Plateau,Mass Market


In [9]:
sales

,sale_id,sale_date,customer_id,product_id,store_id,quantity,unit_price,discount_pct,payment_method
0,1,2026-03-24,C003576,P00189,S017,2,87000,0,Bank Transfer
1,2,2025-11-11,C008570,P00373,S029,1,175000,0,Cash
2,3,2026-01-15,C004439,P00232,S033,2,126000,10,Mobile Money
3,4,2026-07-05,C007041,P00399,S046,1,577000,0,Bank Transfer
4,5,2026-02-26,C007482,P00097,S002,1,204000,15,Card
...,...,...,...,...,...,...,...,...,...
1999995,1999996,2025-03-22,C013690,P00049,S010,1,18000,10,Card
1999996,1999997,2026-06-15,C021601,P00020,S061,3,205000,15,Bank Transfer
1999997,1999998,2025-06-27,C019392,P00002,S007,1,121000,0,Card
1999998,1999999,2025-03-08,C010292,P00146,S013,1,91000,0,Bank Transfer


In [10]:
# date dimension
sales['sale_date'] = pd.to_datetime(sales['sale_date'])

dim_date = pd.DataFrame({
    "full_date": sales['sale_date'].drop_duplicates().sort_values()
}
)

dim_date["date_key"] = (
    dim_date["full_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

dim_date['day'] = dim_date['full_date'].dt.day
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek

In [11]:
dim_date

,full_date,date_key,day,month,year,quarter,day_of_week
279,2025-01-01,20250101,1,1,2025,1,2
620,2025-01-02,20250102,2,1,2025,1,3
1367,2025-01-03,20250103,3,1,2025,1,4
1355,2025-01-04,20250104,4,1,2025,1,5
1154,2025-01-05,20250105,5,1,2025,1,6
...,...,...,...,...,...,...,...
240,2026-08-16,20260816,16,8,2026,3,6
517,2026-08-17,20260817,17,8,2026,3,0
402,2026-08-18,20260818,18,8,2026,3,1
101,2026-08-19,20260819,19,8,2026,3,2


In [12]:
# generate surrogate keys

dim_customer = customers.copy()
dim_customer['customer_key'] = range(1, len(dim_customer) + 1)

In [13]:
dim_customer


,customer_id,customer_name,email,city,state,customer_segment,customer_key
0,C000001,David Okoro,customer000001@quickmart.example,Ikeja,Lagos,Mass Market,1
1,C000002,Peter Garba,customer000002@quickmart.example,Maiduguri,Borno,Mass Market,2
2,C000003,Hauwa Ogunleye,customer000003@quickmart.example,Port Harcourt,Rivers,Mass Market,3
3,C000004,Chinedu Okafor,customer000004@quickmart.example,Maiduguri,Borno,Mass Market,4
4,C000005,Chinedu Musa,customer000005@quickmart.example,Jos,Plateau,Mass Market,5
...,...,...,...,...,...,...,...
24995,C024996,Blessing Lawal,customer024996@quickmart.example,Abeokuta,Ogun,Mass Market,24996
24996,C024997,Kelechi Danladi,customer024997@quickmart.example,Ado-Ekiti,Ekiti,Premium,24997
24997,C024998,Grace Musa,customer024998@quickmart.example,Abuja,FCT,Mass Market,24998
24998,C024999,Esther Ibrahim,customer024999@quickmart.example,Ado-Ekiti,Ekiti,Mass Market,24999


In [14]:
dim_product = products.copy()
dim_product['product_key'] = range(1, len(dim_product) + 1)

In [15]:
dim_store = stores.copy()
dim_store['store_key'] = range(1, len(dim_store) + 1)

In [16]:
dim_sale = sales.copy()
dim_sale['sale_key'] = range(1, len(dim_sale) + 1)

In [17]:
# fact table

fact_sales = sales.copy()

fact_sales = fact_sales.merge(
    dim_customer[['customer_id','customer_key']],
    on='customer_id',
    how='left'
    )
fact_sales = fact_sales.merge(
    dim_product[['product_id','product_key']],
    on='product_id',
    how='left'
    )
fact_sales = fact_sales.merge(
    dim_store[['store_id','store_key']],
    on='store_id',
    how='left'
    )
fact_sales = fact_sales.merge(
    dim_date[['full_date', 'date_key']],
    left_on='sale_date',
    right_on='full_date',
    how='left'
    )


In [18]:
fact_sales

,sale_id,sale_date,customer_id,product_id,store_id,quantity,unit_price,discount_pct,payment_method,customer_key,product_key,store_key,full_date,date_key
0,1,2026-03-24,C003576,P00189,S017,2,87000,0,Bank Transfer,3576,189,17,2026-03-24,20260324
1,2,2025-11-11,C008570,P00373,S029,1,175000,0,Cash,8570,373,29,2025-11-11,20251111
2,3,2026-01-15,C004439,P00232,S033,2,126000,10,Mobile Money,4439,232,33,2026-01-15,20260115
3,4,2026-07-05,C007041,P00399,S046,1,577000,0,Bank Transfer,7041,399,46,2026-07-05,20260705
4,5,2026-02-26,C007482,P00097,S002,1,204000,15,Card,7482,97,2,2026-02-26,20260226
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,1999996,2025-03-22,C013690,P00049,S010,1,18000,10,Card,13690,49,10,2025-03-22,20250322
1999996,1999997,2026-06-15,C021601,P00020,S061,3,205000,15,Bank Transfer,21601,20,61,2026-06-15,20260615
1999997,1999998,2025-06-27,C019392,P00002,S007,1,121000,0,Card,19392,2,7,2025-06-27,20250627
1999998,1999999,2025-03-08,C010292,P00146,S013,1,91000,0,Bank Transfer,10292,146,13,2025-03-08,20250308


In [19]:
fact_sales['sales_amount'] = fact_sales['quantity'] * fact_sales['unit_price']

In [20]:
fact_sales.columns.tolist()

['sale_id',
 'sale_date',
 'customer_id',
 'product_id',
 'store_id',
 'quantity',
 'unit_price',
 'discount_pct',
 'payment_method',
 'customer_key',
 'product_key',
 'store_key',
 'full_date',
 'date_key',
 'sales_amount']

In [21]:
# Select fact column

fact_sales = fact_sales[
    ['sale_id', 'date_key', 'customer_key', 'product_key',
     'store_key', 'quantity', 'unit_price', 'sales_amount']
]

In [22]:
fact_sales

,sale_id,date_key,customer_key,product_key,store_key,quantity,unit_price,sales_amount
0,1,20260324,3576,189,17,2,87000,174000
1,2,20251111,8570,373,29,1,175000,175000
2,3,20260115,4439,232,33,2,126000,252000
3,4,20260705,7041,399,46,1,577000,577000
4,5,20260226,7482,97,2,1,204000,204000
...,...,...,...,...,...,...,...,...
1999995,1999996,20250322,13690,49,10,1,18000,18000
1999996,1999997,20260615,21601,20,61,3,205000,615000
1999997,1999998,20250627,19392,2,7,1,121000,121000
1999998,1999999,20250308,10292,146,13,1,91000,91000


In [25]:
dim_customer.to_csv('dim_customer.csv', index=False)
dim_product.to_csv('dim_product.csv', index=False)
dim_store.to_csv('dim_store.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)
fact_sales.to_csv('fact_sales.csv', index=False)

In [26]:
import os

os.listdir()

['.config',
 'drive',
 'dim_product.csv',
 'fact_sales.csv',
 'dim_customer.csv',
 'dim_date.csv',
 'dim_store.csv',
 'sample_data']

In [27]:
from google.colab import files
files.download('dim_customer.csv')
files.download('dim_product.csv')
files.download('dim_store.csv')
files.download('dim_date.csv')
files.download('fact_sales.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>